# numpyを用いた数値計算４

今回の目標：
- 行列の行列式を計算できるようにする
- 連立方程式を行列を用いて解けるようになる
- カーブフィッティングへの応用ができるようになる．

In [ ]:
import numpy as np

## 動機付け: データサイエンスにおける連立方程式～データから「法則」を発見する～

高校までの数学ではある関数$y=f(x)$が与えられて，その性質やその関数から得られる結果（データ）について議論することが多かった．
一方で，データサイエンスでは，逆の状況を考える．
手元にあるのは**観測データ**だけであり，その背後にある **「法則（数式）」を見出す**ことが最大のミッションになる．

今回はその最も基本的なカーブフィッティングをやってみよう(今日の課題)．

ある観測によって，以下の3つのデータポイント（結果）が得られたとする．
* データ1：$(-1, 0)$
* データ2：$(1, 2)$
* データ3：$(2, 9)$

このデータが，何らかの2次関数 $y = ax^2 + bx + c$ に従っていると仮定する．私たちが知りたいのは，未知のパラメータ $a, b, c$ である．

それぞれのデータを式に代入すると，以下の3つの一次方程式ができる．
1. $a(-1)^2 + b(-1) + c = 0$ すなわち $a - b + c = 0$
2. $a(1)^2 + b(1) + c = 2$ すなわち $a + b + c = 2$
3. $a(2)^2 + b(2) + c = 9$ すなわち $4a + 2b + c = 9$

これは次のように「行列とベクトル」の掛け算の形に直せる．

$$
\begin{pmatrix} 1 & -1 & 1 \\\ 1 & 1 & 1 \\\ 4 & 2 & 1 \end{pmatrix}
\begin{pmatrix} a \\\ b \\\ c \end{pmatrix}
=
\begin{pmatrix} 0 \\\ 2 \\\ 9 \end{pmatrix}
$$

左の行列を $X$，未知のパラメータベクトルを $\theta$（シータ），右の観測結果ベクトルを $y$ と置くと，これは **$X\theta = y$** というシンプルな形になる．
あとはこの$\theta$に関する方程式を解けばよい．

## どうやって$\theta$を求めるか？

### 方法１：逆行列を用いる．

#### 逆行列の定義

正方行列$A$に対して
$AB=BA=I$となる行列$B$のことをAの逆行列といい，$A^{-1}$と表す．
ただし，$I$は単位行列である．

#### 逆行列を用いた解法
$X \theta=y$に対して$X$の逆行列$X^{-1}$を左からかけると
$\theta=X^{-1}y$と$\theta$を求めることができる．

#### 逆行列の存在・非存在

正方行列$A$に対して，いつでも逆行列があるとは限らない．
逆行列の存在・非存在は行列式によって判定できる：
- $A$の行列式が0でない $\iff$ $A$の逆行列が存在する．

行列式の定義については，線形代数の授業を復習してください．
行列式の図形的意味は，
- 2次正方行列の場合は，列ベクトルが作る平行四辺形の面積, 
- 3次正方行列の場合は，列ベクトルが作る平行六面体の体積. 

行列式は`np.linalg.det()`を用いて計算できる．

In [ ]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[1, 2], [2, 4]])
print(np.linalg.det(A))
print(np.linalg.det(B))
# 出力
# -2.0000000000000004
# 0.0

## 逆行列の計算

逆行列が存在するとき，逆行列は`np.linalg.inv()`を用いて計算できる：

In [ ]:
print(np.linalg.inv(A))
# 出力
# [[-2.   1. ]
#  [ 1.5 -0.5]]

逆行列が存在しないときはエラーがでる：

In [ ]:
print(np.linalg.inv(B))
# 出力
# LinAlgError: Singular matrix

### 練習問題
次の行列$A$, $B$の行列式を求めよ．

$$ A=\begin{pmatrix} 2 & 1 \\\ 2 & 3 \end{pmatrix}, 
B= \begin{pmatrix} 34 & 40\\\ -102 & -120 \end{pmatrix}$$

さらに，$X=A$ or $B$に逆行列が存在するときには，$X$の逆行列を求め，連立方程式
$$X\begin{pmatrix}x \\\ y \end{pmatrix}=\begin{pmatrix}1 \\\ 2 \end{pmatrix}$$
を解け．

<details>
<summary>解答例を見る（クリックで展開）</summary>

>```python
>A = np.array([[2, 1], [2, 3]])
>B = np.array([[34, 40], [-102, -120]])
>print('Aの行列式は', np.linalg.det(A))
>print('Bの行列式は', np.linalg.det(B))
># X=Aのときは逆行列が存在し，
>print('Aの逆行列は', np.linalg.inv(A))
>print('X=Aのとき連立方程式の解は', np.linalg.inv(A)@np.array([1,2]))
>```
</details>


### 方法２：`np.linalg.solve`を用いる．

 NumPy の `np.linalg.solve` に任せるだけで，一瞬で連立方程式を解いてくれる．
 （内部的には，線形代数で学んだ掃き出し法を行っている.）
連立方程式$X\theta=y$に対し，
`np.linalg.solve(X, y)`で$\theta$の解が得られる．

In [ ]:
A = np.array([[2,1], [2,3]])
np.linalg.solve(A,np.array([1,2]))
# 出力

### 練習問題
次の連立方程式を`np.linalg.solve`を用いて解け．
$$ 
\begin{cases}
a+b+c=6\\\
2a-b+c=3\\\
a+2b-c=2
\end{cases}
$$

<details>
<summary>解答例を見る（クリックで展開）</summary>

>```python
>C = np.array([[1,1,1],[2,-1,1],[1,2,-1]])
>b = np.array([6,3,2])
>print('連立方程式C \theta = bの解は', np.linalg.solve(C,b))
>```
</details>

## 提出課題
二次関数$y=ax^2+bx+c$は，3点$(-1,0)$, $(1,2)$, $(2,9)$を通るとする．このとき$a,b,c$を求めよ．

## 発展課題
3次関数$y=ax^3+bx^2+cx +d$は4点$(-1,0)$, $(0,2)$, $(1,0)$, $(2,0)$を通るとする．このとき$a,b,c,d$を求めよ．